## 1. Data Preparation


In [1]:
import pandas as pd
from pathlib import Path

In [2]:
data_path = Path("../../data/processed")

train = pd.read_csv(data_path / "train.csv")
val = pd.read_csv(data_path / "val.csv")

print("Training:",train.shape)
print("Validation:",val.shape)

Training: (12178, 26)
Validation: (4941, 26)


In [3]:
train.head()

,Customer ID,snapshot,split,recency,frequency,monetary,avg_order_value,std_order_value,tenure,avg_days_between_orders,...,return_rate,orders_last_30d,orders_last_60d,orders_last_90d,spend_last_90d,spend_trend,frequency_trend,is_uk,weekend_share,churn
0,12346,2010-09-09,train,72.421528,2,169.36,84.680,81.501128,190.452778,118.031250,...,0.000000,0.0,0.0,1.0,142.31,1.000000,1.0,1,0.000000,1
1,12349,2010-09-09,train,132.444444,1,1068.52,1068.520,0.000000,132.444444,47.002778,...,0.022601,0.0,0.0,0.0,0.00,-1.000000,-1.0,0,0.000000,0
2,12355,2010-09-09,train,110.500694,1,488.21,488.210,0.000000,110.500694,47.002778,...,0.000000,0.0,0.0,0.0,0.00,-1.000000,-1.0,0,0.000000,1
3,12358,2010-09-09,train,93.480556,2,1697.93,848.965,821.467161,274.667361,181.186806,...,0.000000,0.0,0.0,0.0,0.00,-1.000000,-1.0,0,0.000000,0
4,12359,2010-09-09,train,78.566667,5,2012.03,402.406,249.260546,277.436111,49.717361,...,0.046719,0.0,0.0,1.0,489.80,0.172612,0.0,0,0.372093,0


### Feature & Target Selection

In [4]:
drop_cols = [
    "Customer ID",
    "snapshot",
    "split",
    "churn"
]

# features
x_train = train.drop(columns=drop_cols)
x_val = val.drop(columns=drop_cols)

# target
y_train = train["churn"]
y_val = val["churn"]

print("X_train:", x_train.shape)
print("y_train:", y_train.shape)
print("features:", x_train.shape[1])


X_train: (12178, 22)
y_train: (12178,)
features: 22


In [5]:
print(y_train.value_counts())
print(y_train.value_counts(normalize=True) * 100)

churn
1    7330
0    4848
Name: count, dtype: int64
churn
1    60.190507
0    39.809493
Name: proportion, dtype: float64


## 2. Model Training & Validation

### 2.1 Logistic Regression Model

#### Model Setup

In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression

In [7]:
lr_model = make_pipeline(
    RobustScaler(),
    LogisticRegression(max_iter=1000)
)

#### Model Training

In [8]:
lr_model.fit(x_train,y_train)

,steps,"[('robustscaler', ...), ('logisticregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,with_centering,True
,with_scaling,True
,quantile_range,"(25.0, ...)"
,copy,True
,unit_variance,False
,penalty,'l2'
,dual,False


#### Validation Predictions

In [9]:
y_val_pred = lr_model.predict(x_val)

y_val_proba = lr_model.predict_proba(x_val)

print("Classes:", lr_model.classes_)
print("Predictions:", y_val_pred[:5])
print("Probabilities:", y_val_proba[:5])

Classes: [0 1]
Predictions: [1 0 0 1 1]
Probabilities: [[5.99041519e-05 9.99940096e-01]
 [6.05171275e-01 3.94828725e-01]
 [5.74856472e-01 4.25143528e-01]
 [2.35166276e-01 7.64833724e-01]
 [2.55826512e-01 7.44173488e-01]]


#### Validation Evaluation

In [10]:
from sklearn.metrics import(
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [11]:
accuracy = accuracy_score(y_val, y_val_pred)
precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))

Accuracy: 0.7838
Precision: 0.8198
Recall: 0.8723
F1 Score: 0.8452


In [12]:
confusion_matrix(y_val, y_val_pred)

array([[ 957,  641],
       [ 427, 2916]])

### 2.2 Decision Tree Model

#### Model Setup

In [13]:
from sklearn.tree import DecisionTreeClassifier

In [14]:
dt_model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=20,
    random_state=42
)

#### Model Training

In [15]:
dt_model.fit(x_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,5
,min_samples_split,2
,min_samples_leaf,20
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


#### Validation Predictions

In [16]:
dt_val_pred = dt_model.predict(x_val)

dt_val_proba = dt_model.predict_proba(x_val)

print("Predictions:", dt_val_pred[:5])
print("Probabilities:", dt_val_proba[:5])

Predictions: [1 0 0 1 1]
Probabilities: [[0.365      0.635     ]
 [0.61295181 0.38704819]
 [0.61295181 0.38704819]
 [0.18350515 0.81649485]
 [0.10741049 0.89258951]]


#### Validation Evaluation

In [17]:
dt_accuracy = accuracy_score(y_val, dt_val_pred)
dt_precision = precision_score(y_val, dt_val_pred)
dt_recall = recall_score(y_val, dt_val_pred)
dt_f1 = f1_score(y_val, dt_val_pred)

print("Accuracy:", round(dt_accuracy, 4))
print("Precision:", round(dt_precision, 4))
print("Recall:", round(dt_recall, 4))
print("F1 Score:", round(dt_f1, 4))


Accuracy: 0.777
Precision: 0.7907
Recall: 0.9118
F1 Score: 0.8469


In [18]:
confusion_matrix(y_val, dt_val_pred)


array([[ 791,  807],
       [ 295, 3048]])

### 2.3 Random Forest Model

#### Model Setup

In [19]:
from sklearn.ensemble import RandomForestClassifier

In [20]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

#### Model Training

In [21]:
rf_model.fit(x_train,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,8
,min_samples_split,2
,min_samples_leaf,20
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


#### Validation Predictions

In [22]:
rf_val_pred = rf_model.predict(x_val)

rf_val_proba = rf_model.predict_proba(x_val)

print("Predictions:", rf_val_pred[:5])
print("Probabilities:", rf_val_proba[:5])


Predictions: [1 0 0 1 1]
Probabilities: [[0.31263971 0.68736029]
 [0.65454016 0.34545984]
 [0.5615765  0.4384235 ]
 [0.25464647 0.74535353]
 [0.15850846 0.84149154]]


#### Validation Evaluation

In [23]:
rf_accuracy = accuracy_score(y_val, rf_val_pred)
rf_precision = precision_score(y_val, rf_val_pred)
rf_recall = recall_score(y_val, rf_val_pred)
rf_f1 = f1_score(y_val, rf_val_pred)

print("Accuracy:", round(rf_accuracy, 4))
print("Precision:", round(rf_precision, 4))
print("Recall:", round(rf_recall, 4))
print("F1 Score:", round(rf_f1, 4))


Accuracy: 0.7877
Precision: 0.8004
Recall: 0.9141
F1 Score: 0.8535


In [24]:
confusion_matrix(y_val, rf_val_pred)

array([[ 836,  762],
       [ 287, 3056]])

## 3. Average Precision (AP)

In [25]:
from sklearn.metrics import average_precision_score

lr_scores = lr_model.predict_proba(x_val)[:, 1]
dt_scores = dt_model.predict_proba(x_val)[:, 1]
rf_scores = rf_model.predict_proba(x_val)[:, 1]

lr_ap = average_precision_score(y_val, lr_scores)
dt_ap = average_precision_score(y_val, dt_scores)
rf_ap = average_precision_score(y_val, rf_scores)

print("Logistic Regression AP:", round(lr_ap, 4))
print("Decision Tree AP:", round(dt_ap, 4))
print("Random Forest AP:", round(rf_ap, 4))


Logistic Regression AP: 0.8882
Decision Tree AP: 0.8655
Random Forest AP: 0.8911


## 4. Threshold Tuning

### 4.1 Random Forest

In [26]:
thresholds = [0.3, 0.35, 0.4, 0.45, 0.5]

results = []

In [27]:
for threshold in thresholds:

    y_pred = (rf_scores >= threshold).astype(int)

    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)

    results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })


In [28]:
results_df = pd.DataFrame(results)

print(results_df.round(4))


   Threshold  Precision  Recall      F1
0       0.30     0.7461  0.9818  0.8478
1       0.35     0.7573  0.9725  0.8515
2       0.40     0.7696  0.9623  0.8552
3       0.45     0.7852  0.9426  0.8567
4       0.50     0.8004  0.9141  0.8535


#### Best Threshold & Validation Results

In [29]:
best_index = results_df["F1"].idxmax()

best_threshold = results_df.loc[best_index, "Threshold"]

print("Best Threshold:", best_threshold)
print("Best F1:", results_df.loc[best_index, "F1"])


Best Threshold: 0.45
Best F1: 0.856715606307776


In [30]:
rf_best_pred = (rf_scores >= best_threshold).astype(int)

print("Precision:", precision_score(y_val, rf_best_pred))
print("Recall:", recall_score(y_val, rf_best_pred))
print("F1:", f1_score(y_val, rf_best_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, rf_best_pred))


Precision: 0.7851981061549963
Recall: 0.9425665569847442
F1: 0.856715606307776

Confusion Matrix:
[[ 736  862]
 [ 192 3151]]


### 4.2 Logistic Regression

In [31]:
thresholds = [0.3, 0.35, 0.4, 0.45, 0.5]

lr_results = []

In [32]:
for threshold in thresholds:

    # Convert probabilities to classes
    lr_pred = (lr_scores >= threshold).astype(int)

    # Calculate metrics
    precision = precision_score(y_val, lr_pred)
    recall = recall_score(y_val, lr_pred)
    f1 = f1_score(y_val, lr_pred)

    # Save results
    lr_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })


In [33]:
lr_results_df = pd.DataFrame(lr_results)

display(lr_results_df.round(4))

,Threshold,Precision,Recall,F1
0,0.30,0.7628,0.9560,0.8485
1,0.35,0.7783,0.9387,0.8510
2,0.40,0.7911,0.9198,0.8506
3,0.45,0.8068,0.9007,0.8512
4,0.50,0.8198,0.8723,0.8452


### 4.3 Decision Tree

In [34]:
dt_scores = dt_model.predict_proba(x_val)[:, 1]

thresholds = [0.3, 0.35, 0.4, 0.45, 0.5]

dt_results = []


In [35]:
for threshold in thresholds:

    dt_pred = (dt_scores >= threshold).astype(int)

    precision = precision_score(y_val, dt_pred)
    recall = recall_score(y_val, dt_pred)
    f1 = f1_score(y_val, dt_pred)

    dt_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

In [36]:
dt_results_df = pd.DataFrame(dt_results)

display(dt_results_df.round(4))

,Threshold,Precision,Recall,F1
0,0.30,0.7548,0.9686,0.8484
1,0.35,0.7548,0.9686,0.8484
2,0.40,0.7774,0.9339,0.8485
3,0.45,0.7890,0.9171,0.8483
4,0.50,0.7907,0.9118,0.8469


## 5. Model Comparison & Selection

In [37]:
lr_best = lr_results_df.loc[lr_results_df["F1"].idxmax()]
dt_best = dt_results_df.loc[dt_results_df["F1"].idxmax()]
rf_best = results_df.loc[results_df["F1"].idxmax()]

comparison = pd.DataFrame([
    {"Model": "Logistic Regression", **lr_best.to_dict()},
    {"Model": "Decision Tree", **dt_best.to_dict()},
    {"Model": "Random Forest", **rf_best.to_dict()}
])

comparison = comparison.sort_values(
    by="F1",
    ascending=False
)

display(comparison.round(4))


,Model,Threshold,Precision,Recall,F1
2,Random Forest,0.45,0.7852,0.9426,0.8567
0,Logistic Regression,0.45,0.8068,0.9007,0.8512
1,Decision Tree,0.40,0.7774,0.9339,0.8485


## 6. Final Model Training

### Load Full Training Data

In [38]:
full_train = pd.read_csv(data_path / "full_train.csv")

full_train.shape


(17119, 27)

### Feature Count Check

In [39]:
X_full_train = full_train.drop(columns=drop_cols)

y_full_train = full_train["churn"]

print("Features :", X_full_train.shape)
print("Target :", y_full_train.shape)


Features : (17119, 23)
Target : (17119,)


### Extra Column Check

In [40]:
print("Original Features:", x_train.shape)
print("Full Train Features:", X_full_train.shape)

print("Extra Columns:")
print(X_full_train.columns.difference(x_train.columns).tolist())

print("Missing Columns:")
print(x_train.columns.difference(X_full_train.columns).tolist())


Original Features: (12178, 22)
Full Train Features: (17119, 23)
Extra Columns:
['partition']
Missing Columns:
[]


### Final Feature Alignment

In [41]:
X_full_train = full_train[x_train.columns].copy()

y_full_train = full_train["churn"]

print("Original Features:", x_train.shape)
print("Final Training Features:", X_full_train.shape)

print("Same Columns:", X_full_train.columns.equals(x_train.columns))


Original Features: (12178, 22)
Final Training Features: (17119, 22)
Same Columns: True


### Train Final Random Forest

In [42]:
final_rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

final_rf_model.fit(X_full_train, y_full_train)

,n_estimators,100
,criterion,'gini'
,max_depth,8
,min_samples_split,2
,min_samples_leaf,20
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [43]:
final_rf_model.n_features_in_

22

## 7. Final Test Evaluation

### Load Test Data

In [44]:
test = pd.read_csv(data_path / "test.csv")

X_test = test[x_train.columns].copy()
y_test = test["churn"]

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)
print("Same Features:", X_test.columns.equals(X_full_train.columns))


X_test: (5253, 22)
y_test: (5253,)
Same Features: True


### Test Predictions

In [45]:
final_test_scores = final_rf_model.predict_proba(X_test)[:, 1]

final_threshold = 0.45

final_test_pred = (final_test_scores >= final_threshold).astype(int)

print("Probabilities:", final_test_scores[:5])
print("Predictions:", final_test_pred[:5])


Probabilities: [0.76164353 0.22777527 0.57082358 0.79796903 0.88093575]
Predictions: [1 0 1 1 1]


### Test Metrics & Confusion Matrix

In [46]:
final_accuracy = accuracy_score(y_test, final_test_pred)
final_precision = precision_score(y_test, final_test_pred)
final_recall = recall_score(y_test, final_test_pred)
final_f1 = f1_score(y_test, final_test_pred)
final_ap = average_precision_score(y_test, final_test_scores)

print("FINAL TEST RESULTS")
print("Accuracy:", round(final_accuracy, 4))
print("Precision:", round(final_precision, 4))
print("Recall:", round(final_recall, 4))
print("F1 Score:", round(final_f1, 4))
print("Average Precision:", round(final_ap, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, final_test_pred))


FINAL TEST RESULTS
Accuracy: 0.6994
Precision: 0.662
Recall: 0.9587
F1 Score: 0.7832
Average Precision: 0.8224

Confusion Matrix:
[[ 822 1456]
 [ 123 2852]]


## 8. Churn Distribution Comparison

In [47]:

print("Train Churn Distribution:")
print(train["churn"].value_counts(normalize=True))

print("\nValidation Churn Distribution:")
print(val["churn"].value_counts(normalize=True))

print("\nTest Churn Distribution:")
print(test["churn"].value_counts(normalize=True))


Train Churn Distribution:
churn
1    0.601905
0    0.398095
Name: proportion, dtype: float64

Validation Churn Distribution:
churn
1    0.676584
0    0.323416
Name: proportion, dtype: float64

Test Churn Distribution:
churn
1    0.566343
0    0.433657
Name: proportion, dtype: float64


## 9. Model Saving & Loading

### Model Package

In [48]:
import joblib

model_package = {
    "model": final_rf_model,
    "threshold": final_threshold,
    "feature_columns": X_full_train.columns.tolist(),
    "positive_class": 1
}


### Save Model

In [49]:
models_path = Path("../../models/prediction")

models_path.mkdir(parents=True, exist_ok=True)

joblib.dump(
    model_package,
    models_path / "churn_model.joblib"
)

print("Model saved successfully")


Model saved successfully


### Load Saved Model

In [50]:
package = joblib.load("../../models/prediction/churn_model.joblib")

model = package["model"]
threshold = package["threshold"]
feature_columns = package["feature_columns"]
positive_class = package["positive_class"]

print("Model loaded successfully!")
print("Threshold:", threshold)
print("Number of Features:", len(feature_columns))


Model loaded successfully!
Threshold: 0.45
Number of Features: 22


## 10. Churn Prediction Function

In [51]:
def predict_churn(customer_data):

    customer_df = pd.DataFrame([customer_data])

    missing = [
        col for col in feature_columns
        if col not in customer_df.columns
    ]

    if missing:
        raise ValueError(f"Missing features: {missing}")

    customer_df = customer_df[feature_columns]

    churn_index = list(model.classes_).index(positive_class)

    probability = model.predict_proba(customer_df)[0, churn_index]

    prediction = int(probability >= threshold)

    return {
        "churn_probability": round(float(probability), 4),
        "churn_prediction": prediction
    }


### Single-Customer Prediction Test

In [52]:
sample_customer = X_test.iloc[0].to_dict()

result = predict_churn(sample_customer)

print(result)


{'churn_probability': 0.7616, 'churn_prediction': 1}
